In [1]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [2]:
connection_url = URL.create(
    "mysql+mysqlconnector",
    username = "user",
    password = "your_password",
    host = "localhost",
    database = "swiftdrop_analytics"
)

In [3]:
engine = create_engine(connection_url)

In [4]:
tables = ['orders', 'products', 'deliveries']
dfs = {}

for table in tables:
    query = f"select * from {table}"
    dfs[table] = pd.read_sql(query, con=engine)

In [5]:
for table_name, df in dfs.items():
    print(f"--- Table: {table_name} ---")
    display(df.head())
    print("\n")


--- Table: orders ---


,order_id,customer_id,product_id,order_date,discount_applied,order_status
0,ORD10001,C5093,P1044,2026-03-23 05:54:00,0.05,Delivered
1,ORD10002,C6030,P1068,2026-02-16 17:09:00,0.05,Delivered
2,ORD10003,C6368,P1045,2026-02-05 07:25:00,0.00,Delivered
3,ORD10004,C4561,P1042,2026-03-19 11:36:00,0.05,Delivered
4,ORD10005,C2643,P1065,2026-03-17 22:09:00,0.00,Delivered




--- Table: products ---


,product_id,category,product_name,cogs,retail_price,measure_value,measure_unit
0,P1001,Apparel,Men's T-Shirt,1288.17,2824.36,1.0,pcs
1,P1002,Home Decor,Ceramic Vase,403.97,853.31,2.0,unit
2,P1003,Personal Care,Face Wash,1071.03,1934.42,3.0,unit
3,P1004,Home Decor,Throw Blanket,1179.22,1471.50,4.0,unit
4,P1005,Home Decor,Throw Blanket,896.71,1237.96,5.0,unit




--- Table: deliveries ---


,delivery_id,order_id,warehouse_id,dispatch_time,delivery_time,shipping_cost
0,D20000,ORD10001,WH-South,2026-03-23 06:54:00,2026-03-23 09:24:00,111.48
1,D20001,ORD10002,WH-Central,2026-02-16 17:25:00,2026-02-16 19:33:00,87.93
2,D20002,ORD10003,WH-North,2026-02-05 08:09:00,2026-02-05 11:21:00,100.14
3,D20003,ORD10004,WH-Central,2026-03-19 12:10:00,2026-03-19 12:51:00,66.33
4,D20004,ORD10005,WH-North,2026-03-17 23:00:00,2026-03-18 02:14:00,103.37


### Remove extra spaces from order_statues column

In [6]:
orders_df = dfs["orders"]

In [7]:
orders_df["order_status"] = orders_df["order_status"].str.strip()
orders_df.head()

,order_id,customer_id,product_id,order_date,discount_applied,order_status
0,ORD10001,C5093,P1044,2026-03-23 05:54:00,0.05,Delivered
1,ORD10002,C6030,P1068,2026-02-16 17:09:00,0.05,Delivered
2,ORD10003,C6368,P1045,2026-02-05 07:25:00,0.00,Delivered
3,ORD10004,C4561,P1042,2026-03-19 11:36:00,0.05,Delivered
4,ORD10005,C2643,P1065,2026-03-17 22:09:00,0.00,Delivered


### 1.  Total Net Revenue.

In [8]:
orders_df = dfs["orders"]
products_df = dfs["products"]
merge_dfs = pd.merge(orders_df, products_df, on = "product_id", how = "inner")
merge_dfs.head()
total_revenue = merge_dfs.apply(
    lambda row: row["retail_price"] * (1 - row["discount_applied"]), axis = 1).sum().round(2)
total_revenue = total_revenue / 1000000
print(f"Total revenue is: {total_revenue: .2f}M")

Total revenue is:  7.13M


### 2. Total Net Revenue by each category. 

In [9]:
orders_df = dfs["orders"]
products_df = dfs["products"]
merge_dfs = pd.merge(orders_df, products_df, on = "product_id", how = "inner")
revenue_by_category = merge_dfs.groupby("category").apply(
    lambda row: (row["retail_price"] * ( 1 - row["discount_applied"])).sum()).round(2).reset_index().rename(columns = {0: "total_revenue"}) 
revenue_by_category

,category,total_revenue
0,Apparel,1291497.59
1,Electronics,1129013.97
2,Groceries,1229939.27
3,Home Decor,1602197.73
4,Personal Care,1874721.81


###  3. Absolute Gross Profit (profit after substracting the cost of goods and shipping cost)

In [10]:
orders_df = dfs["orders"]
products_df = dfs["products"]
deliveries_df = dfs["deliveries"]

merge_dfs = pd.merge(orders_df, products_df, on="product_id", how="inner")
merge_dfs = pd.merge(merge_dfs, deliveries_df, on="order_id", how="inner")

merge_dfs["profit"] = (
    (merge_dfs["retail_price"] * (1 - merge_dfs["discount_applied"])) 
    - merge_dfs["cogs"] 
    - merge_dfs["shipping_cost"]
)

delivered_df = merge_dfs[merge_dfs["order_status"] == "Delivered"]
total_profit = (delivered_df["profit"].sum()/1000000).round(2)
print(f"Total absolute gross profit: {total_profit}M") 


Total absolute gross profit: 2.25M


### 4. Profit Margin % per Category.

In [11]:
orders_df = dfs["orders"]
products_df = dfs["products"]
deliveries_df = dfs["deliveries"]

merge_dfs = pd.merge(orders_df, products_df, on="product_id", how="inner")
merge_dfs = pd.merge(merge_dfs, deliveries_df, on="order_id", how="inner")

delivered_df = merge_dfs[merge_dfs["order_status"] == "Delivered"]

delivered_df["total_revenue"] = delivered_df["retail_price"] * ( 1 - delivered_df["discount_applied"])
    
delivered_df["profit"] = (
delivered_df["total_revenue"] - delivered_df["cogs"] - delivered_df["shipping_cost"]
)

profit_margin = (delivered_df.groupby("category").agg(
total_profit = ("profit", "sum"),
total_revenue = ("total_revenue", "sum")
).assign(profit_margin = lambda x: (x["total_profit"] / x["total_revenue"] * 100)
.round(2)).reset_index().sort_values(by = "profit_margin", ascending = False)
                )
profit_margin[["category", "profit_margin"]]


,category,profit_margin
4,Personal Care,42.88
3,Home Decor,39.23
0,Apparel,35.87
1,Electronics,32.12
2,Groceries,31.36


### 5. Revenue Lost to Cancellations.

In [12]:
orders_df = dfs["orders"]
products_df = dfs["products"]
deliveries_df = dfs["deliveries"]

merge_dfs = pd.merge(orders_df, products_df, on="product_id", how="inner")
merge_dfs = pd.merge(merge_dfs, deliveries_df, on="order_id", how="inner")

delivered_df = merge_dfs[merge_dfs["order_status"] == "Cancelled"]
delivered_df["total_revenue"] = delivered_df["retail_price"] * ( 1 - delivered_df["discount_applied"])
loss = delivered_df["total_revenue"].sum() / 1000
print(f"Revenue Lost to Cancellations.: {loss: .2f}k") 


Revenue Lost to Cancellations.:  728.33k


### 6. Which product sold the most in each category 

In [13]:
orders_df = dfs["orders"]
products_df = dfs["products"]

merge_dfs = pd.merge(orders_df, products_df, on = "product_id", how ="inner")

product_sold = merge_dfs[merge_dfs["order_status"] == "Delivered"]

product_counts = (
    product_sold.groupby(["category", "product_name"]).size().reset_index(name = "order_count")
)


most_sold = (
    product_counts.sort_values("order_count", ascending = False).groupby("category").head(1).reset_index(drop = True)
)
most_sold

,category,product_name,order_count
0,Personal Care,Electric Toothbrush,300
1,Groceries,Almonds,271
2,Home Decor,Throw Blanket,260
3,Apparel,Cotton Socks,238
4,Electronics,Bluetooth Speaker,207


### 7. In which month product sold the most?


In [14]:
orders_df = dfs["orders"]
products_df = dfs["products"]
merge_dfs = pd.merge(orders_df, products_df, on = "product_id", how = "inner")


merge_dfs["month_num"] = merge_dfs["order_date"].dt.month

merge_dfs["month_name"] = merge_dfs["order_date"].dt.strftime("%b")

product_sold = merge_dfs[merge_dfs["order_status"] == "Delivered"]

product_counts = (
    product_sold.groupby(["month_name", "month_num", "product_name", "category"])["order_id"].count()
        .reset_index(name = "order_count").sort_values(by = "order_count", ascending = False)
        .drop(columns = "month_num")
)
product_counts
most_sold_products = (
    product_counts.groupby(["product_name", "category"]).head(1)
)
most_sold_products

,month_name,product_name,category,order_count
31,Jan,Electric Toothbrush,Personal Care,105
50,Mar,Almonds,Groceries,102
28,Jan,Coffee Beans,Groceries,97
45,Jan,Sunscreen,Personal Care,93
29,Jan,Cotton Socks,Apparel,91
21,Feb,Throw Blanket,Home Decor,91
63,Mar,Moisturizer,Personal Care,91
61,Mar,LED Strip Lights,Home Decor,82
51,Mar,Bluetooth Speaker,Electronics,82
34,Jan,Green Tea,Groceries,79


### 8. Checks if orders with >15% discounts are resulting in a net loss after shipping.

In [15]:
orders_df = dfs["orders"]
products_df = dfs["products"]
deliveries_df = dfs["deliveries"]

merge_dfs = pd.merge(orders_df, products_df, on="product_id", how="inner")
merge_dfs = pd.merge(merge_dfs, deliveries_df, on="order_id", how="inner")

ordered_product = merge_dfs[
(merge_dfs["discount_applied"] > 0.15) &
    (merge_dfs["order_status"] == "Delivered")]

ordered_product["total_revenue"] = (
   ordered_product["retail_price"] * (1 - ordered_product["discount_applied"])
)
ordered_product["net_profit"] = (
    ordered_product["total_revenue"] - ordered_product["cogs"] - ordered_product["shipping_cost"]
)

total_net = ordered_product["net_profit"].sum().round(2)
print(f"Orders with >15% discounts are resulting: {total_net}")

Orders with >15% discounts are resulting: 137297.33


###  9. Delivery Time (Order to Door).

In [16]:
orders_df = dfs["orders"]
deliveries_df = dfs["deliveries"]
merge_dfs = pd.merge(orders_df, deliveries_df, on="order_id", how="inner")

ordered = merge_dfs[merge_dfs["order_status"] == "Delivered"]
ordered["order_to_door"] = ordered["delivery_time"] - ordered["order_date"]
ordered[["order_date", "delivery_time", "order_to_door"]].head(5)

,order_date,delivery_time,order_to_door
0,2026-03-23 05:54:00,2026-03-23 09:24:00,0 days 03:30:00
1,2026-02-16 17:09:00,2026-02-16 19:33:00,0 days 02:24:00
2,2026-02-05 07:25:00,2026-02-05 11:21:00,0 days 03:56:00
3,2026-03-19 11:36:00,2026-03-19 12:51:00,0 days 01:15:00
4,2026-03-17 22:09:00,2026-03-18 02:14:00,0 days 04:05:00


### 10. Which warehouse is the fastest at getting packages out to the riders? 

In [17]:
orders_df = dfs["orders"]
deliveries_df = dfs["deliveries"]
merge_dfs = pd.merge(orders_df, deliveries_df, on="order_id", how="inner")

merge_dfs["warehouse_get_packages"] = merge_dfs["dispatch_time"] - merge_dfs["order_date"]

warehouse_get_packages = merge_dfs[["warehouse_id", "warehouse_get_packages"]]
warehouse_get_packages.sort_values("warehouse_get_packages", ascending = True).head(5)

,warehouse_id,warehouse_get_packages
3215,WH-South,0 days 00:15:00
30,WH-West,0 days 00:15:00
4204,WH-East,0 days 00:15:00
4839,WH-Central,0 days 00:15:00
1265,WH-East,0 days 00:15:00


### 11. Lists orders that took longer than 4 hours( Late Delivery Detection).

In [18]:
orders_df = dfs["orders"]
deliveries_df = dfs["deliveries"]
merge_dfs = pd.merge(orders_df, deliveries_df, on="order_id", how="inner")

ordered = merge_dfs[merge_dfs["order_status"] == "Delivered"]
ordered["order_to_door"] = (ordered["delivery_time"] - ordered["order_date"])
late_delivery = ordered[ordered["order_to_door"] >  "0 days 04:00:00"]
late_delivery.head(5)


,order_id,customer_id,product_id,order_date,discount_applied,order_status,delivery_id,warehouse_id,dispatch_time,delivery_time,shipping_cost,order_to_door
4,ORD10005,C2643,P1065,2026-03-17 22:09:00,0.0,Delivered,D20004,WH-North,2026-03-17 23:00:00,2026-03-18 02:14:00,103.37,0 days 04:05:00
8,ORD10009,C7314,P1039,2026-03-01 15:16:00,0.1,Delivered,D20008,WH-West,2026-03-01 16:26:00,2026-03-01 20:09:00,66.63,0 days 04:53:00
13,ORD10014,C3358,P1039,2026-02-14 02:55:00,0.0,Delivered,D20013,WH-North,2026-02-14 04:22:00,2026-02-14 07:18:00,55.49,0 days 04:23:00
14,ORD10015,C8459,P1023,2026-03-26 02:54:00,0.2,Delivered,D20014,WH-West,2026-03-26 03:20:00,2026-03-26 08:19:00,60.75,0 days 05:25:00
16,ORD10017,C6457,P1056,2026-03-29 16:18:00,0.0,Delivered,D20016,WH-North,2026-03-29 18:17:00,2026-03-29 20:25:00,57.65,0 days 04:07:00


### 12. Are we spending too much on delivery relative to the product price(Shipping Cost as % of Revenue)?

In [41]:
orders_df = dfs["orders"]
products_df = dfs["products"]
deliveries_df = dfs["deliveries"]

merge_dfs = pd.merge(orders_df, products_df, on="product_id", how="inner")
merge_dfs = pd.merge(merge_dfs, deliveries_df, on="order_id", how="inner")
merge_dfs["reven"] = merge_dfs["retail_price"] * ( 1 - merge_dfs["discount_applied"])

ship_sum =    merge_dfs.groupby("category")["shipping_cost"].sum()
rev_sum =    merge_dfs.groupby("category")["reven"].sum()

ship_cost_ratio = (
(ship_sum / rev_sum ) * 100
).round(2).reset_index(name = "ship_cost_ratio")

ship_cost_ratio

,category,ship_cost_ratio
0,Apparel,6.01
1,Electronics,5.19
2,Groceries,6.13
3,Home Decor,4.69
4,Personal Care,4.73


### 13. Peak Hour Traffic.

In [92]:
orders_dfs = dfs["orders"]
orders_dfs["hour"] = orders_dfs["order_date"].dt.strftime("%I %p")
peak_hour = orders_dfs.groupby("hour")["order_id"].count().reset_index(name = "order_count").sort_values("order_count", ascending = False)  
peak_hour.head(5) 


,hour,order_count
13,07 PM,229
2,02 AM,226
22,12 AM,220
21,11 PM,219
1,01 PM,219


### 14. Top 5 Most Returned Products.

In [97]:
orders_df = dfs["orders"]
products_df = dfs["products"]
merge_dfs = pd.merge(orders_df, products_df, on="product_id", how="inner")

returned_product = merge_dfs[merge_dfs["order_status"] == "Returned"]
top_5_returned_product = returned_product.groupby("product_name")["order_id"].count().reset_index(name = "returned").sort_values("returned", ascending = False)
top_5_returned_product.head(5)

,product_name,returned
4,Cotton Socks,18
11,LED Strip Lights,16
13,Moisturizer,14
3,Coffee Beans,14
6,Electric Toothbrush,14


 ### 15. Average Order Value (AOV) by Customer.

In [110]:
orders_df = dfs["orders"]
products_df = dfs["products"]
merge_dfs = pd.merge(orders_df, products_df, on="product_id", how="inner")

merge_dfs["rev"] = (merge_dfs["retail_price"] * ( 1 - merge_dfs["discount_applied"]))

order_total= merge_dfs.groupby(["customer_id", "order_id"])["rev"].sum().reset_index(name = "total_order")
AOV = order_total.groupby("customer_id")["total_order"].mean().reset_index(name = "AOV").round(2)
AOV.head(5)

,customer_id,AOV
0,C1000,1507.56
1,C1001,1418.64
2,C1002,855.48
3,C1007,1590.91
4,C1008,1594.69
